# Passive Distance and Relative Radial Velocity Estimation — Prototype

This notebook implements and runs all experiments for the thesis *Lightweight
Distance and Relative Radial Velocity Estimation with a Passive RF Receiver*. It
runs on the RFSoC 4x2 under PYNQ, using a modified RFSoC-MTS overlay (`mts.py`)
for capture.

The receiver listens to an unsynchronized continuous-wave transmitter at
2.41 GHz and estimates:

1. **Distance** — from received signal strength (RSSI), via a calibrated
   log-distance path-loss model.
2. **Relative radial velocity** — from the Doppler shift of the received tone,
   using Kay's weighted phase-difference estimator.

The notebook is organized into: hardware setup and RFDC configuration, shared
utility functions, the RSSI/distance pipeline, and the Doppler/velocity pipeline
(including the stationary null test and the clock-drift characterization that
together establish the velocity error floor).

> **Hardware required.** These cells only run on an RFSoC 4x2 with the overlay
> loaded and a transmitter active at 2.41 GHz. They cannot be run offline.

## Setup

Load the overlay, synchronize the converter tiles (MTS), and verify the clock
tree. The first capture below is taken *before* configuring the digital
down-converter, as a raw sanity check that samples are arriving on both
channels.

In [ ]:
from mts import mtsOverlay
from pynq import allocate
import numpy as np
import matplotlib.pyplot as plt
import time
import xrfdc

In [ ]:
ol = mtsOverlay('mts.bit')

In [ ]:
ol.init_tile_sync()
ol.verify_clock_tree()

In [ ]:
nonAlignedCaptureSamples = np.zeros((2, len(ol.adc_capture_chA_I)), dtype=np.int16)
ol.internal_capture(nonAlignedCaptureSamples)

In [ ]:
plot_start = 0
plot_num_samples = 256

fig, axes = plt.subplots(nrows=1, ncols=1)
axes.stem(nonAlignedCaptureSamples[0][plot_start:plot_start+plot_num_samples])
axes.stem(nonAlignedCaptureSamples[1][plot_start:plot_start+plot_num_samples], linefmt='r:', markerfmt='rx')
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(nrows=2, ncols=1)
axes[0].plot(nonAlignedCaptureSamples[0][plot_start:plot_start+plot_num_samples])
axes[1].plot(nonAlignedCaptureSamples[1][plot_start:plot_start+plot_num_samples])
plt.show()

### RFDC digital down-conversion

Configure the ADC fine mixer (NCO) on the connected tiles to down-convert the
2.41 GHz input to complex baseband (real-to-complex mode, Nyquist zone 2). After
this, each capture yields an I/Q baseband pair centered near DC. `debug_tiles()`
in the next cell prints the tile/block status to confirm the mixer and Nyquist
settings took effect.

In [ ]:
NCO_FREQ = 2410.0
NYQUIST_ZONE = 2

rfdc = ol.usp_rf_data_converter_1

for tile in [0, 2]:
    for block in [0, 1]:
        rfdc.adc_tiles[tile].blocks[block].NyquistZone = NYQUIST_ZONE
        rfdc.adc_tiles[tile].blocks[block].MixerSettings['MixerMode'] = xrfdc.MIXER_MODE_R2C
        rfdc.adc_tiles[tile].blocks[block].MixerSettings['MixerType'] = xrfdc.MIXER_TYPE_FINE
        rfdc.adc_tiles[tile].blocks[block].MixerSettings['CoarseMixFreq'] = xrfdc.COARSE_MIX_BYPASS
        rfdc.adc_tiles[tile].blocks[block].MixerSettings['Freq'] = NCO_FREQ
        rfdc.adc_tiles[tile].blocks[block].MixerSettings['PhaseOffset'] = 0.0
        rfdc.adc_tiles[tile].blocks[block].MixerSettings['EventSource'] = xrfdc.EVNT_SRC_TILE
        rfdc.adc_tiles[tile].blocks[block].MixerSettings['FineMixerScale'] = xrfdc.MIXER_SCALE_1P0
        rfdc.adc_tiles[tile].blocks[block].UpdateEvent(xrfdc.EVENT_MIXER)

In [ ]:
def debug_tiles():
    """
    Print out the status and settings of all ADC and DAC blocks.
    This is useful for debugging and verifying that the blocks are configured as expected.
    """
    for ti, tile in enumerate(rfdc.adc_tiles):
        print("ADC tile", ti)
        for bi, block in enumerate(tile.blocks):
            try:
                print("  block", bi, "NyquistZone:", block.NyquistZone)
                print("  block", bi, "BlockStatus:", block.BlockStatus)
                print("  block", bi, "MixerSettings:", block.MixerSettings)
            except Exception as e:
                print("  block", bi, "not enabled or inaccessible:", e)
    for ti, tile in enumerate(rfdc.dac_tiles):
        print("DAC tile", ti)
        for bi, block in enumerate(tile.blocks):
            try:
                print("  block", bi, "NyquistZone:", block.NyquistZone)
                print("  block", bi, "BlockStatus:", block.BlockStatus)
                print("  block", bi, "MixerSettings:", block.MixerSettings)
            except Exception as e:
                print("  block", bi, "not enabled or inaccessible:", e)

#debug_tiles()

## Utility functions

Shared signal-processing helpers used by both pipelines. `get_baseband_fast`
takes a raw I/Q capture, removes the DC offset, and decimates by 100 in two
polyphase stages (`resample_poly`, which fuses anti-alias FIR filtering and
decimation). This reduces the 4 GS/s stream to a 40 MS/s baseband rate, enough
to resolve the small Doppler shifts of interest while keeping processing light.

In [ ]:
from scipy.signal import resample_poly

FS_BASEBAND = 4e9 / 100
F_CARRIER = 2.41e9
C = 299_792_458

def get_baseband_fast(I_raw, Q_raw):
    """
    Convert raw I/Q samples to complex baseband and perform decimation.

    Args:
        I_raw (np.ndarray): Raw I samples as int16.
        Q_raw (np.ndarray): Raw Q samples as int16.
    
    Returns:
        np.ndarray: Complex baseband samples after decimation.
    """
    x = I_raw.astype(np.float32) + 1j * Q_raw.astype(np.float32)
    x -= x.mean()
    x = resample_poly(x, 1, 10)
    x = resample_poly(x, 1, 10)
    return x

## RSSI and distance estimation

Distance is estimated from received power using a calibrated log-distance
path-loss model:

  RSSI(d) = RSSI(d_ref) − 10·n·log10(d / d_ref)

The workflow is: (1) measure RSSI at several known distances to build a
calibration set, (2) fit the path-loss exponent *n* and reference level by least
squares, then (3) invert the model to estimate distance from a new RSSI
measurement. RSSI is reported in dBFS (relative to ADC full scale); each
measurement averages many captures to reduce noise.

The first cell below is a quick saturation check — confirm `max |I|` and
`max |Q|` are well below full scale (8192) so the ADC is not clipping.

In [ ]:
capture = np.zeros((2, len(ol.adc_capture_chA_I)), dtype=np.int16)
ol.internal_capture(capture)
print("max |I|:", np.abs(capture[0]).max())
print("max |Q|:", np.abs(capture[1]).max())

In [ ]:
FULL_SCALE = 8192 # 14-bit ADC

def compute_rssi(x_bb, trim_fraction=0.05):
    """
    Compute RSSI in dBFS from baseband samples, optionally trimming the start and end to avoid transients.

    Args:
        x_bb (np.ndarray): Complex baseband samples.
        trim_fraction (float): Fraction of samples to trim from the start and end. Default is 0.05.

    Returns:
        float: RSSI in dBFS.
    """
    n = len(x_bb)
    trim = int(n * trim_fraction)
    x_steady = x_bb[trim:n-trim] if trim > 0 else x_bb

    power_avg = np.mean(np.abs(x_steady) ** 2)
    rssi_db = 10 * np.log10(power_avg / (FULL_SCALE ** 2) + 1e-30) # Add small constant to avoid log of zero

    return rssi_db

def measure_rssi_once():
    """
    Take a single capture and return the computed RSSI in dBFS.

    Returns:
        float: RSSI in dBFS for the captured signal.
    """
    capture = np.zeros((2, len(ol.adc_capture_chA_I)), dtype=np.int16)
    ol.internal_capture(capture)
    x_bb = get_baseband_fast(capture[0], capture[1])
    return compute_rssi(x_bb)

def measure_rssi(num_captures=20, verbose=False):
    """
    Take multiple captures and return the average RSSI and its standard deviation.

    Args:
        num_captures (int): Number of captures to take for averaging. Default is 20.
        verbose (bool): If True, print the RSSI of each individual capture. Default is False.
    
    Returns:
        tuple: (mean RSSI in dBFS, standard deviation of RSSI in dB)
    """
    rssi_values = []
    for i in range(num_captures):
        rssi = measure_rssi_once()
        rssi_values.append(rssi)
        if verbose:
            print(f"Capture {i+1}/{num_captures}: {rssi:.2f} dBFS")

    rssi_arr = np.array(rssi_values)
    return rssi_arr.mean(), rssi_arr.std()

#measure_rssi(num_captures=10, verbose=True)

### Calibration

`calibrate_at_distance(d)` prompts you to place the receiver at a known distance
and records the mean/std RSSI there. Build up several points across the range of
interest, then fit the model. The helper cells below allow inspecting, clearing,
or manually overriding the calibration set — the hard-coded `calibration_data`
list records the exact points used in the reported experiments so the fit is
reproducible without re-running the live measurements.

In [ ]:
# Storage for calibration measurements
calibration_data = [] # List of (distance_m, rssi_mean, rssi_std)

def calibrate_at_distance(distance_m, num_captures=20):
    """
    Take a calibration measurement at a known distance.
    Call this multiple times at different distances to build up the calibration dataset.

    Args:
        distance_m (float): The known distance in meters at which to take the calibration measurement.
        num_captures (int): Number of captures to take for averaging the RSSI. Default is 20.
    
    Returns:
        tuple: (mean RSSI in dBFS, standard deviation of RSSI in dB) for the given distance.
    """
    print(f">>> Calibrating at d = {distance_m} m")
    print(f"    Make sure RX is exactly {distance_m} m from TX. Press Enter when ready...")
    input()

    print(f"    Measuring ({num_captures} captures)...")
    rssi_mean, rssi_std = measure_rssi(num_captures=num_captures)

    print(f"    Result: RSSI = {rssi_mean:.3f} ± {rssi_std:.3f} dBFS")

    calibration_data.append((distance_m, rssi_mean, rssi_std))
    return rssi_mean, rssi_std

def clear_calibration():
    """
    Clear all stored calibration measurements.
    """
    calibration_data.clear()
    print("Calibration data cleared.")

def show_calibration():
    """
    Print out the current calibration measurements in a readable format.
    """
    if not calibration_data:
        print("No calibration data yet.")
        return

    print(f"{'Distance (m)':>12}  {'RSSI (dBFS)':>14}  {'Std (dB)':>10}")
    for d, r, s in sorted(calibration_data):
        print(f"{d:>12.3f}  {r:>14.3f}  {s:>10.3f}")

In [ ]:
#clear_calibration()

In [ ]:
#calibrate_at_distance(1.0, num_captures=100)

In [ ]:
show_calibration()

### Fitting and distance estimation

Fit the path-loss model to the calibration points, then estimate distance from a
fresh RSSI reading. The fit reports the recovered exponent *n* and the
goodness of fit; estimation propagates the RSSI uncertainty into a distance
uncertainty.

In [ ]:
path_loss_model = {
    'fitted': False,
    'n_exponent': None,
    'rssi_ref': None,
    'd_ref': None,
}

def fit_path_loss_model(d_ref=1.0):
    """
    Fit the log-distance path-loss model to the calibration data.
    Model: RSSI(d) = RSSI(d_ref) - 10 * n * log10(d / d_ref)
    With multiple calibration points, fits n via least squares (regression of RSSI on log10(distance)).

    Args:
        d_ref (float): Reference distance in meters (1 m is conventional). Default is 1.0 m.
    """
    if len(calibration_data) < 2:
        print(f"Need at least 2 calibration points. Have {len(calibration_data)}.")
        return None

    distances = np.array([d for d, _, _ in calibration_data])
    rssis = np.array([r for _, r, _ in calibration_data])

    # Linear regression: RSSI = a + b * log10(d)
    log_d = np.log10(distances)
    slope, intercept = np.polyfit(log_d, rssis, 1)

    n_exp = -slope / 10
    rssi_ref = slope * np.log10(d_ref) + intercept

    # Calculate R² for fit quality
    rssi_pred = slope * log_d + intercept
    ss_res = np.sum((rssis - rssi_pred) ** 2)
    ss_tot = np.sum((rssis - rssis.mean()) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 1.0

    path_loss_model.update({
        'fitted': True,
        'n_exponent': n_exp,
        'rssi_ref': rssi_ref,
        'd_ref': d_ref,
        'r2': r2,
    })

    print(f"Fitted path-loss model:")
    print(f"  Path-loss exponent n = {n_exp:.3f}")
    print(f"  RSSI at d_ref = {d_ref} m: {rssi_ref:.2f} dBFS")
    print(f"  R² = {r2:.4f}")

def estimate_distance(rssi, std=0.0):
    """
    Estimate distance from an RSSI measurement, using the fitted model.

    Args:
        rssi (float): The measured RSSI in dBFS for which to estimate the distance.
        std (float): The standard deviation of the RSSI measurement. Default is 0.0.
    
    Returns:
        tuple: (estimated distance in meters, uncertainty in distance in meters)
    """
    if not path_loss_model['fitted']:
        print("Path-loss model not fitted yet. Call fit_path_loss_model() first.")
        return None

    n_exp = path_loss_model['n_exponent']
    rssi_ref = path_loss_model['rssi_ref']
    d_ref = path_loss_model['d_ref']

    d_est = d_ref * 10 ** ((rssi_ref - rssi) / (10 * n_exp))
    d_std = d_est * np.log(10) / (10 * n_exp) * std if std > 0 else 0.0
    return d_est, d_std

fit_path_loss_model()

In [ ]:
def measure_and_estimate_distance(num_captures=30):
    """
    Take multiple captures to measure RSSI and its uncertainty, then estimate distance with uncertainty propagation.

    Args:
        num_captures (int): Number of captures to take for averaging the RSSI. Default is 30.
    """
    rssi, std = measure_rssi(num_captures=num_captures)
    d_est, d_std = estimate_distance(rssi, std)

    print(f"RSSI:               {rssi:.2f} ± {std:.2f} dBFS")
    print(f"Estimated distance: {d_est:.3f} ± {d_std:.3f} m")

measure_and_estimate_distance(num_captures=100)

## Doppler frequency and velocity estimation

Radial velocity is estimated from the Doppler shift of the received tone. The
baseband frequency is estimated with **Kay's weighted phase-difference
estimator** (`freq_kay`), which is near the Cramér–Rao bound for a single tone
and requires no FFT or buffering.

Because the transmitter and receiver run on independent, unsynchronized clocks,
the measured baseband frequency contains an unknown clock offset in addition to
the Doppler shift. Absolute frequency therefore cannot be turned into absolute
velocity directly; instead the velocity is recovered from frequency *differences*
across a short motion window, in which the slowly-varying clock offset largely
cancels (the clock-offset bracketing used in `measure_velocity`).

This section contains three experiments:

- **Velocity measurement** — `measure_velocity` captures the tone
  before, during, and after motion and converts the Doppler shift to radial
  velocity.
- **Null test** — repeats the measurement with everything stationary. Since the
  true velocity is zero, the spread of these estimates gives the empirical
  velocity-precision floor of the pipeline.
- **Clock-drift characterization** — measures the stability of the baseband
  frequency over time and across capture lengths, which is what ultimately
  limits the velocity accuracy.

In [ ]:
START_OFFSET = 32 * (1 << 10)

def freq_kay(x, fs):
    """
    Estimate the frequency of a complex baseband signal using Kay's frequency estimator.

    Args:
        x (np.ndarray): Complex baseband samples.
        fs (float): Sampling frequency of the baseband signal.
    
    Returns:
        float: Estimated frequency in Hz.
    """
    x = x - x.mean()
    N = len(x)
    if N < 3:
        return 0.0

    dphi = np.angle(x[1:] * np.conj(x[:-1]))
    n = np.arange(N - 1)
    w = 6.0 * n * (N - 1 - n) / (N * (N**2 - 1))
    return fs * np.sum(w * dphi) / (2 * np.pi)

def _estimate_freqs_from_buffers(bufs_I, bufs_Q, num_samples):
    """
    Copy out and process a list of captured buffer pairs, returning Kay's
    frequency estimate for each. Kept separate from the capture phase so that
    the (slow) copy-out and processing happen outside any drift-sensitive window.

    Args:
        bufs_I (list): Captured I buffers (PL DRAM).
        bufs_Q (list): Captured Q buffers (PL DRAM).
        num_samples (int): Number of samples to process from each buffer.

    Returns:
        np.ndarray: Estimated frequency in Hz for each buffer pair.
    """
    offsets_I = [(buf.physical_address - ol.ddr4_0.base_address) // 2 for buf in bufs_I]
    offsets_Q = [(buf.physical_address - ol.ddr4_0.base_address) // 2 for buf in bufs_Q]

    freqs = np.zeros(len(bufs_I))
    for k in range(len(bufs_I)):
        I_raw = np.array(ol.adc_dram_capture[(offsets_I[k] + START_OFFSET):(offsets_I[k] + num_samples)])
        Q_raw = np.array(ol.adc_dram_capture[(offsets_Q[k] + START_OFFSET):(offsets_Q[k] + num_samples)])
        x_bb = get_baseband_fast(I_raw, Q_raw)
        freqs[k] = freq_kay(x_bb, FS_BASEBAND)
        del I_raw, Q_raw, x_bb

    return freqs

def measure_velocity(n_avg=4, num_samples=(16 << 20), instant=False):
    """
    Measure the radial velocity of the receiver by performing multiple captures
    before, during, and after motion, and applying Kay's frequency estimator to each.

    The function preallocates a separate DMA buffer pair for every capture to minimize the time
    spent on copying data out of the buffers, allowing for a fast capture phase.
    
    Args:
        n_avg (int): Number of captures to average for each of the three phases (before, motion, after). Default is 4.
        num_samples (int): Number of samples to capture in each buffer. Default is 16M samples.
        instant (bool): If True, do not wait for user input before each phase. Default is False.
    
    Returns:
        float: Estimated radial velocity in m/s.
    """
    if n_avg < 1:
        print("n_avg must be at least 1.")
        raise ValueError("n_avg must be at least 1.")

    total = 3 * n_avg

    # Preallocate all buffer pairs upfront
    print(f"Allocating {total} buffer pairs ({total * 2 * num_samples * 2 // (1 << 20)} MB PL DRAM)...")
    bufs_I = [allocate(num_samples, dtype=np.int16, target=ol.ddr4_0) for _ in range(total)]
    bufs_Q = [allocate(num_samples, dtype=np.int16, target=ol.ddr4_0) for _ in range(total)]

    times = []

    try:
        # ---- PHASE 1: capture only (fast, drift-sensitive window) ----
        t0 = time.monotonic()

        print(">>> STATIC reference BEFORE — keep still. Press Enter.")
        if not instant: input()
        for k in range(n_avg):
            ol.dram_capture(bufs_I[k], bufs_Q[k])
            times.append(time.monotonic() - t0)

        print(">>> MOTION — move now. Press Enter.")
        if not instant: input()
        for k in range(n_avg, 2*n_avg):
            ol.dram_capture(bufs_I[k], bufs_Q[k])
            times.append(time.monotonic() - t0)

        print(">>> STATIC reference AFTER — keep still. Press Enter.")
        if not instant: input()
        for k in range(2*n_avg, 3*n_avg):
            ol.dram_capture(bufs_I[k], bufs_Q[k])
            times.append(time.monotonic() - t0)

        capture_window = time.monotonic() - times[0]
        print(f"\nCapture window (drift-sensitive): {capture_window:.2f} s")

        # ---- PHASE 2 + 3: copy out + process (slow, but drift no longer matters) ----
        print("Copying out and processing...")
        freqs = _estimate_freqs_from_buffers(bufs_I, bufs_Q, num_samples)

    finally:
        for buf in bufs_I: buf.freebuffer()
        for buf in bufs_Q: buf.freebuffer()
        del bufs_I, bufs_Q

    freqs = np.array(freqs)
    times = np.array(times)

    f_before = freqs[:n_avg].mean()
    t_before = times[:n_avg].mean()

    f_motion = freqs[n_avg:2*n_avg].mean()
    t_motion = times[n_avg:2*n_avg].mean()

    f_after = freqs[2*n_avg:3*n_avg].mean()
    t_after = times[2*n_avg:3*n_avg].mean()

    frac = (t_motion - t_before) / (t_after - t_before)
    f_interp = f_before + frac * (f_after - f_before)
    residual = f_motion - f_interp
    v = -C * residual / F_CARRIER

    print(f"\nBefore: {f_before:.1f} @ {t_before:.2f}s | Motion: {f_motion:.1f} @ {t_motion:.2f}s | After: {f_after:.1f} @ {t_after:.2f}s")
    print(f"Drift: {(f_after-f_before)/(t_after-t_before):+.2f} Hz/s | Residual: {residual:+.2f} Hz")
    print(f"Velocity: {v:+.3f} m/s")

    return v

### Null test (stationary baseline)

With the transmitter and receiver held stationary, the true radial velocity is
zero, so any non-zero estimate reflects estimator and clock-induced uncertainty.
The standard deviation over these trials is the baseline precision against which
the motion results are compared.

In [ ]:
NUM_NULL_TESTS = 10

nulls = np.zeros(NUM_NULL_TESTS)
print(f"Running null test {NUM_NULL_TESTS} times. Keep everything stationary throughout.\n")
for trial in range(NUM_NULL_TESTS):
    print(f"\n=== Trial {trial+1}/{NUM_NULL_TESTS} ===")
    nulls[trial] = measure_velocity(n_avg=4, num_samples=(16 << 20), instant=True)

print(f"\n=== Null distribution over {NUM_NULL_TESTS} trials ===")
print(f"Mean:  {nulls.mean():+.3f} m/s")
print(f"Std:   {nulls.std():.3f} m/s")
print(f"Range: {nulls.min():+.2f} to {nulls.max():+.2f} m/s")
print(f"All:   {np.round(nulls, 3)}")

In [ ]:
measure_velocity()

### Clock-drift characterization

These cells measure how the estimated tone frequency varies over time with both
devices stationary, since this frequency instability is what ultimately bounds
the velocity accuracy. `capture_drift_session` captures a series of estimates at
a target cadence, `summarize_drift` computes a session's mean, standard
deviation, drift-corrected (residual) standard deviation, and fitted drift rate,
and `run_drift_experiment` repeats a session over several trials.

Two regimes are examined: a slow run (captures every 2 s over ~2 min, showing
longer-term wander) and fast back-to-back runs (showing short-term stability and
the offset spread between runs).

In [ ]:
PROC_CHUNK = 10 # Process this many captures at a time (PS-DRAM safety)

def capture_drift_session(num_captures, target_interval_s, num_samples=(16 << 20)):
    """
    Run a single drift-measurement session, capturing num_captures at the specified target interval.

    Args:
        num_captures (int): Number of captures to take in the session.
        target_interval_s (float): Target spacing between captures in seconds.
        num_samples (int): Number of samples to capture in each buffer. Default is 16M samples.
    
    Returns:
        tuple: (times, freqs) where times is an array of capture timestamps and freqs is an array of estimated frequencies
    """
    bufs_I = [allocate(num_samples, dtype=np.int16, target=ol.ddr4_0) for _ in range(num_captures)]
    bufs_Q = [allocate(num_samples, dtype=np.int16, target=ol.ddr4_0) for _ in range(num_captures)]

    times = np.zeros(num_captures)
    freqs = np.zeros(num_captures)

    try:
        # ---- Capture phase: throttle toward the target cadence ----
        t0 = time.monotonic()
        for k in range(num_captures):
            sleep_for = (t0 + k * target_interval_s) - time.monotonic()
            if sleep_for > 0:
                time.sleep(sleep_for)
            ol.dram_capture(bufs_I[k], bufs_Q[k])
            times[k] = time.monotonic() - t0

        achieved = np.mean(np.diff(times)) if num_captures > 1 else 0.0
        print(f"{num_captures} captures | Target: {target_interval_s*1e3:.0f} ms | "
              f"Achieved: {achieved*1e3:.1f} ms | Total: {times[-1]:.1f} s")

        # ---- Copy out + process in chunks (frees PS arrays per chunk) ----
        for c0 in range(0, num_captures, PROC_CHUNK):
            c1 = min(c0 + PROC_CHUNK, num_captures)
            freqs[c0:c1] = _estimate_freqs_from_buffers(bufs_I[c0:c1], bufs_Q[c0:c1], num_samples)
            print(f"Processed captures {c0}..{c1-1}")
    finally:
        for buf in bufs_I: buf.freebuffer()
        for buf in bufs_Q: buf.freebuffer()
        del bufs_I, bufs_Q

    return times, freqs

def summarize_drift(times, freqs):
    """
    Summarize a drift session: mean, raw std, drift-corrected (residual) std,
    and the fitted linear drift rate.

    Args:
        times (np.ndarray): Capture timestamps in seconds.
        freqs (np.ndarray): Estimated frequency per capture in Hz.

    Returns:
        dict: {mean, std, resid_std, drift} in Hz (drift in Hz/s).
    """
    slope, intercept = np.polyfit(times, freqs, 1)
    resid_std = (freqs - (slope * times + intercept)).std()
    return {'mean': freqs.mean(), 'std': freqs.std(), 'resid_std': resid_std, 'drift': slope}

def run_drift_experiment(num_captures, target_interval_s, num_trials, num_samples=(16 << 20)):
    """
    Run a drift-measurement experiment consisting of multiple trials, each with a specified number of captures.

    Args:
        num_captures (int): Captures per trial.
        target_interval_s (float): Target spacing between captures in seconds.
        num_trials (int): Number of trials to run.
        num_samples (int): Samples per capture. Default is 16M.

    Returns:
        list: One (times, freqs) tuple per trial.
    """
    trials = []
    for trial in range(num_trials):
        print(f"\n=== Trial {trial+1}/{num_trials} ===")
        times, freqs = capture_drift_session(num_captures, target_interval_s, num_samples)
        s = summarize_drift(times, freqs)
        print(f"Mean: {s['mean']:.1f} Hz | Std: {s['std']:.2f} Hz | "
              f"Residual: {s['resid_std']:.2f} Hz | Drift: {s['drift']:+.2f} Hz/s")
        trials.append((times, freqs))

    return trials

In [ ]:
def moving_average(x, w):
    """
    Compute a centered moving average of the input array x with window size w.

    Args:
        x (np.ndarray): Input array.
        w (int): Window size for the moving average.
    
    Returns:
        np.ndarray: Array of moving average values, length len(x) - w + 1.
    """
    return np.convolve(x, np.ones(w) / w, mode='valid')

def plot_trials(trials, ma_window=None):
    """
    Plot frequency vs time for all trials on one axis.

    Args:
        trials (list): List of (times, freqs) tuples.
        ma_window (int): If set, overlay a centered moving average of this window on each trial. Default is None.
    """
    plt.figure(figsize=(9, 5))
    cmap = plt.get_cmap('tab10')

    for i, (times, freqs) in enumerate(trials):
        color = cmap(i % 10)
        t = np.asarray(times, float)
        f = np.asarray(freqs, float)

        if ma_window:
            plt.plot(t, f, '-', color=color, lw=0.5, alpha=0.35)
            if len(f) >= ma_window:
                f_ma = moving_average(f, ma_window)
                off = (ma_window - 1) // 2
                plt.plot(t[off:off + len(f_ma)], f_ma, '-', color=color, lw=1.4, label=f"Trial {i+1}")
        else:
            plt.plot(t, f, '-o', color=color, markersize=2, lw=0.6, label=f"Trial {i+1}")

    plt.xlabel("time [s]")
    plt.ylabel("frequency [Hz]")
    plt.grid(True, alpha=0.3)
    plt.legend(fontsize=7, ncol=2)
    plt.tight_layout()
    plt.show()

In [ ]:
# Run a slow drift experiment: 60 captures spaced 2 s apart, over 2 min
trials_slow = run_drift_experiment(num_captures=60, target_interval_s=2.0, num_trials=1)
plot_trials(trials_slow)

In [ ]:
# Run a fast drift experiment: 50 consecutive captures with no spacing
trials_fast = run_drift_experiment(num_captures=50, target_interval_s=0.0, num_trials=6)
plot_trials(trials_fast)

### Capture-length sweep

`sweep_capture_durations` runs a drift session at several capture lengths and
reports raw and drift-corrected (residual) standard deviation for each. This is
used to select the operating capture length: long enough to reach the
clock-limited floor, but no longer than necessary.

In [ ]:
FS_ADC = 4e9 # ADC sampling frequency

def sweep_capture_durations(sample_counts, num_captures=60, target_interval_s=0.0):
    """
    Run a drift session at each capture length and report the raw and
    drift-corrected (residual) standard deviation of the frequency estimates.
    Using the same num_captures and interval at each length keeps
    the total observation time comparable across lengths.

    Args:
        sample_counts (list): Capture lengths (samples per channel) to sweep.
        num_captures (int): Captures per session. Default is 60.
        target_interval_s (float): Target spacing between captures. Default is 0.
    """
    header = f"{'samples':>12} {'dur [ms]':>9} {'raw std':>9} {'resid std':>10} {'drift [Hz/s]':>13}"
    print(header)
    print("-" * len(header))

    for num_samples in sample_counts:
        dur_ms = num_samples / FS_ADC * 1e3
        times, freqs = capture_drift_session(num_captures, target_interval_s, num_samples)
        s = summarize_drift(times, freqs)
        print(f"{num_samples:>12} {dur_ms:>9.2f} {s['std']:>9.2f} "
              f"{s['resid_std']:>10.2f} {s['drift']:>13.3f}")

sweep_capture_durations([8 << 20, 9 << 20, 16 << 20, 30 << 20])